# **Callbacks in CrewAI**

## In CrewAI, a callback is a function that is automatically executed at a specific point during an agent or task’s lifecycle.

## Callbacks can be used to observe execution, log events, inspect or modify outputs, track metrics, handle errors, or trigger additional actions. For example, a task callback can be invoked after a task finishes, allowing the application to capture the task output and perform post-processing without changing the task’s main logic.

## This makes callbacks useful for**monitoring, observability, debugging**, and **integrating custom application logic** into a CrewAI workflow.

## Install Libraries

In [2]:
!pip install -q crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.5/195.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.2/833.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

## Set API Keys for LLMs & Tools

In [3]:
from google.colab import userdata
import os

os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
# os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY_NEW')

## Import necessary classes/methods from the package

In [4]:
from crewai import Agent, Crew, Task, Process, LLM, Process
from crewai_tools import SerperDevTool, DirectoryReadTool
from pydantic import BaseModel
from typing import List


#------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")
#------------------------------------------------------------------------


## Define the LLM Object

In [5]:
# Create an LLM with a temperature of 0 to ensure deterministic outputs from the LLM
# Set temperature to higher values for creative outputs from the LLM

# OPENAI LLMs
llm = LLM(
          model="gpt-5.4-nano",
          base_url="https://api.openai.com/v1",
          api_key = os.environ["OPENAI_API_KEY"],
          temperature=0.7)


# # GROQ hosted LLMs
# llm = LLM(
#     #  model="qwen/qwen3.6-27b",
#      model="openai/gpt-oss-120b",
#      base_url="https://api.groq.com/openai/v1",
#      api_key=os.environ["GROQ_API_KEY"],
#      temperature=0.7)


## Define Tools

In [9]:
#------------------------------------------------------------------------
# Create tools
search_tool = SerperDevTool()  # Search capability
docs_tool = DirectoryReadTool(directory='./blog-posts')  # Reads from local files


## Define Model Classes

In [10]:
#------------------------------------------------------------------------
# Define the Output Class to ensure Structured output from the crew
# This will be used to validate the output of the tasks

class ResearchFindings(BaseModel):
    main_points: List[str]
    key_technologies: List[str]
    societal_impact: str

class Report(BaseModel):
    title: str
    introduction: str
    body: str
    conclusion: str

## Create Agents


In [14]:
researcher = Agent(
    role='Research Analyst',
    goal='Use available tool to collect information and provide up-to-date technical and social analysis on a given topic',
    backstory='An expert analyst with a keen eye for technical nitty-gritty with a perspective on human vakues.',
    tools=[search_tool],
    llm=llm,
    verbose=False
)


#------------------------------------------------------------------------------------------------------------------

writer = Agent(
    role='Content Writer',
    goal='Craft engaging report about the provided topic',
    backstory='A skilled writer with a passion for technology and its impact on humanity.',
    tools=[docs_tool],
    llm=llm,
    verbose=False
)


## Define Callbacks for Tasks

In [15]:
def research_task_callback(output):
    print("\n--- Research Task Completed ---")
    print("Research Output Type:", type(output))
    print("Research Output:")
    print(output.model_dump_json(indent=2))


#---------------------------------------------------

def writing_task_callback(output):
    print("\n--- Writing Task Completed ---")
    print("Writing Output Type:", type(output))
    print("Writing Output:")
    print(output.model_dump_json(indent=2))


## Define Tasks

In [16]:

research_task = Task(
    description='Research the latest trends in the topic {topic}',
    expected_output=('A summary of recent developments including a unique perspective on their significance.'
        'Your output should contain the following:'
        'main_points: the main textual summary of the report'
        'key_technologies: key technologies enabling the change'
        'societal_impact: how it impacts the life of people and the society as a whole'),
    agent=researcher,
    callback=research_task_callback,
    output_pydantic = ResearchFindings
)

#------------------------------------------------------------------------------------------------------------------

writing_task = Task(
    description=("""Write an engaging report about a topic based on the research analyst's summary.
                    You will receive research output in JSON format from the researcher.
                    You need to extract the following piece of information from the object returned by the `research_task`
                    'main_points: the main textual summary of the report'
                    'key_technologies: key technologies enabling the change'
                    'societal_impact: how it impacts the life of people and the society as a whole'
                     'Use this information to write a comprehensive and engaging report.'
                     """),
    expected_output=(
        "A structured report with title, introduction, body, and conclusion, "
        "written in a clear and engaging style."),
    agent=writer,
    callback=writing_task_callback,
    output_pydantic=Report,       # Structured output format
    output_file='blog-posts/report.md',  # The final blog post will be saved here
    # depends_on=[research_task],
    context = [research_task], # Pass research context to writer
    verbose=True
)


## Step Callback for the Crew


In [17]:
def crew_step_callback(output):
    """A sample step callback function."""
    print("--- CREW CALLBACK: STEP COMPLETED ---")
    print(output)
    print("----------------------")


## Define the Crew

In [19]:
# Assemble a crew with planning enabled
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    verbose=False,
    process=Process.sequential,
    # planning=True,  # Enable planning feature
    step_callback=crew_step_callback  # to be executed after every step of the Crew
)


## Run the Crew

In [23]:

results = await crew.kickoff_async(
    inputs={"topic": "Social media and its impact on humans"}
    )

# results

--- CREW CALLBACK: STEP COMPLETED ---
AgentFinish(thought='', output='{\n  "main_points": [\n    "First, algorithms could contribute to increasing depression, anxiety, loneliness, body dissatisfaction, and even suicides by facilitating unhealthy social ...",\n    "This year\'s survey also highlights emerging challenges in the form of AI platforms and chatbots, which we have asked about for the first time.",\n    "The results show that influencers can successfully correct misperceptions on a politically non-polarized topic (mental health).",\n    "The DSA empowers citizens by strengthening the protection of their fundamental rights online and giving them greater control and more choices when they navigate ...",\n    "Live Shopping Experiences: Platforms like TikTok and Instagram are hosting live events where creators showcase products, answer questions in ...",\n    "Social commerce is changing the way we shop. Explore key trends, platforms, and strategies to leverage this growing ecomm

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## What's in results?

In [21]:
#------------------------------------------------------------------------
print("\n\n--- What's in Results ---\n")
obj_attr = dir(results)
for att in obj_attr:
    print(att)
print("\n--------------------------------------\n")

print("\n++++++++++++++++++++++++++++++++++++++++++++++++\n")
print("\n--- What's in Results Token Usage---")
print("\n Usage Tokens: ", results.token_usage)
print("\n--------------------------------------\n")

print("\n++++++++++++++++++++++++++++++++++++++++++++++++\n")
print("\n--- What's in Results output [1]/ Writing Task ---\n")
writing_task_output = results.tasks_output[1]  # Task executed second i.e., writing_task
print("Task Description:", writing_task_output.description)
print("\n--------------------------------------------------------")
# report = writing_task_output.pydantic # OR the next statement means the same
report = results.pydantic # Results/Output of the Last Task executed by the Crew, i.e., writing_task
                          # `results` hold the information about the LAST EXECUTED task which is writing task
print("\n--- What's in Results Tasks Output[1] / Pydantic---")
print(f"\n Report Title: {report.title}")
print(f"\n Report Introduction: {report.introduction}")
print(f"\n Report Body: {report.body}")
print(f"\n Report Conclusion: {report.conclusion}")

print("\n++++++++++++++++++++++++++++++++++++++++++++++++\n")

print("\n--- What's in Results Task output [0]/ Research Task ---\n")
research_task_output = results.tasks_output[0]  # Task executed first i.e., research_task
# print(dir(research_task_output))
print("Task Description:", research_task_output.description)
print("\n--------------------------------------------------------")
research = research_task_output.pydantic
# print(type(research))
print(f"\n Main Points: {research.main_points}")
print(f"\n Key Technologies: {research.key_technologies}")
print(f"\n Societal Impact: {research.societal_impact}")

print("\n++++++++++++++++++++++++++++++++++++++++++++++++\n")




--- What's in Results ---

__abstractmethods__
__annotations__
__class__
__class_getitem__
__class_vars__
__copy__
__deepcopy__
__delattr__
__dict__
__dir__
__doc__
__eq__
__fields__
__fields_set__
__format__
__ge__
__get_pydantic_core_schema__
__get_pydantic_json_schema__
__getattr__
__getattribute__
__getitem__
__getstate__
__gt__
__hash__
__init__
__init_subclass__
__iter__
__le__
__lt__
__module__
__ne__
__new__
__pretty__
__private_attributes__
__pydantic_complete__
__pydantic_computed_fields__
__pydantic_core_schema__
__pydantic_custom_init__
__pydantic_decorators__
__pydantic_extra__
__pydantic_fields__
__pydantic_fields_set__
__pydantic_generic_metadata__
__pydantic_init_subclass__
__pydantic_on_complete__
__pydantic_parent_namespace__
__pydantic_post_init__
__pydantic_private__
__pydantic_root_model__
__pydantic_serializer__
__pydantic_setattr_handlers__
__pydantic_validator__
__reduce__
__reduce_ex__
__replace__
__repr__
__repr_args__
__repr_name__
__repr_recursion__
__repr

## Verify successful context passing between the agents

In [22]:

#------------------------------------------------------------
# Verify successful context passing between the agents
#------------------------------------------------------------

# Extract the outputs of individual tasks
research_output = results.tasks_output[0].pydantic
writing_output = results.tasks_output[1].pydantic

# --- Verification Logic ---
# Check if the research output is a valid Pydantic model
if not isinstance(research_output, ResearchFindings):
    print("❌ Research task did not produce a valid ResearchFindings object.")
else:
    print("✅ Research task produced a valid ResearchFindings object.")

    # Get a specific piece of information from the research findings
    # For example, the first main point or a key technology
    key_research_point = research_output.main_points[0] if research_output.main_points else ""
    key_technology = research_output.key_technologies[0] if research_output.key_technologies else ""

    print(f"\nKey research point to check: '{key_research_point}'")
    print(f"\nKey technology to check: '{key_technology}'")

    # Check if the writing output is a valid Pydantic model
    if not isinstance(writing_output, Report):
        print("❌ Writing task did not produce a valid Report object.")
    else:
        print("✅ Writing task produced a valid Report object.")

        # Now, verify if the writer's report contains the information from the researcher
        # This is the core of the verification
        if key_research_point in writing_output.body or key_research_point in writing_output.introduction:
            print("🎉 SUCCESS: The writer's report successfully incorporated the research context!")
        else:
            print("⚠️ The writer's report seems to be missing the key research context.\n\n")



✅ Research task produced a valid ResearchFindings object.

Key research point to check: 'Mainstream platforms are increasingly using generative AI and automation to produce, moderate, and personalize content—changing what people see, how quickly misinformation can spread, and how easily synthetic media can be mistaken for real.'

Key technology to check: 'Generative AI for content creation, recommendation-support tooling, and automated moderation/safety systems (including AI systems that can amplify or mis-handle harmful content).'
✅ Writing task produced a valid Report object.
🎉 SUCCESS: The writer's report successfully incorporated the research context!


# **DEMO: Agent Callback, Task Callback, Crew Callback**

In [10]:
from crewai import Agent, Task, Crew


# ---------------------------------------------------------
# 1. AGENT CALLBACK
# ---------------------------------------------------------

def agent_callback(output):
    """
    Called after an agent step.

    Useful for:
    - observing reasoning/tool activity
    - logging intermediate results
    - monitoring agent behavior
    """

    print("\n[AGENT CALLBACK]")
    # print("Agent step completed:")
    print(f"Agent '{output.name}' completed")
    print(f"Output: {output.raw}")


# ---------------------------------------------------------
# 2. TASK CALLBACK
# ---------------------------------------------------------

def research_task_callback(output):
    """
    Called when a particular task completes.

    Useful for:
    - validating output
    - storing task results
    - measuring task-level performance
    """

    print("\n[TASK CALLBACK]")
    print("Task completed.")
    print(f"Task '{output.name}' completed")
    print(f"Output: {output.raw}")


    # Example validation
    if not output:
        print("WARNING: Empty task output!")

    # In a real application:
    # database.save(task_output)


# ---------------------------------------------------------
# 3. CREW CALLBACK
# ---------------------------------------------------------

def crew_callback(crew_output):
    """
    Called when the overall crew execution completes.

    Useful for:
    - generating execution summaries
    - recording overall metrics
    - updating dashboards
    """

    print("\n[CREW CALLBACK]")
    print("Crew execution completed.")
    print(f"Task '{crew_output.name}' completed")
    print(f"Output: {crew_output.raw}")


    # In a real application:
    # observability_db.save_run(crew_output)


# ---------------------------------------------------------
# AGENTS
# ---------------------------------------------------------

research_agent = Agent(
    name="Research Analyst",
    role="Market Research Analyst",
    goal="Research the company's market position and competitive landscape.",
    backstory="You are an experienced equity research analyst.",
    step_callback=agent_callback,
    llm=llm
)

financial_agent = Agent(
    name="Financial Analyst",
    role="Financial Analyst",
    goal="Analyze the company's financial performance.",
    backstory="You specialize in financial statement analysis.",
    step_callback=agent_callback,
    llm=llm
)

risk_agent = Agent(
    name="Risk Analyst",
    role="Risk Analyst",
    goal="Identify major investment risks.",
    backstory="You specialize in identifying financial and business risks.",
    step_callback=agent_callback,
    llm=llm
)


# ---------------------------------------------------------
# TASKS
# ---------------------------------------------------------

market_task = Task(
    name="Market Research Task",
    description="""
    Research NVIDIA's current market position,
    major competitors, competitive advantages,
    and major growth opportunities.
    """,
    expected_output="A structured market analysis.",
    agent=research_agent,
    callback=research_task_callback
)

financial_task = Task(
    name="Financial Analysis Task",
    description="""
    Analyze NVIDIA's recent financial performance,
    including revenue growth, profitability and
    important financial trends.
    """,
    expected_output="A structured financial analysis.",
    agent=financial_agent,
    callback=research_task_callback
)

risk_task = Task(
    name="Risk Analysis Task",
    description="""
    Identify the major risks associated with
    investing in NVIDIA.
    """,
    expected_output="A structured risk analysis.",
    agent=risk_agent,
    callback=research_task_callback
)


# ---------------------------------------------------------
# CREW
# ---------------------------------------------------------

crew = Crew(
    name="Investment Research Crew",
    agents=[
        research_agent,
        financial_agent,
        risk_agent
    ],

    tasks=[
        market_task,
        financial_task,
        risk_task
    ],

    # Crew-level callback concept
    callback=crew_callback
)


In [11]:
result = await crew.kickoff_async()


[TASK CALLBACK]
Task completed.
Task 'Market Research Task' completed
Output: ## 1) Executive Market Position (NVIDIA Today)

**NVIDIA’s market position is dominant in accelerated computing**, particularly for:
- **AI training and inference** (data center GPUs, networking, and software stack)
- **High-performance computing (HPC)** (simulation, scientific computing, digital twins)
- **Edge AI** (Jetson and related platforms)
- **AI networking** (InfiniBand / Ethernet-based interconnect and NVIDIA networking software)

**How NVIDIA wins the market**
1. **End-to-end platform**: GPU hardware + high-performance interconnect + optimized software libraries (CUDA ecosystem) + reference architectures.
2. **Developer lock-in**: CUDA and the surrounding toolchain/libraries are deeply embedded across the AI stack.
3. **System-level performance**: Not just faster chips—NVIDIA sells **integrated systems** that reduce time-to-train and improve throughput at scale.
4. **Supply and execution**: NVIDIA

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
print("\nFINAL INVESTMENT RESEARCH")
print(result)


FINAL INVESTMENT RESEARCH
## Major Investment Risks Associated with NVIDIA (NVDA) — Structured Risk Analysis

### 1) Concentration & Demand Cycle Risk (AI Capex / Hyperscaler Spending)
**Description**  
NVIDIA’s growth is highly tied to large-scale AI infrastructure build-outs (training and increasingly inference). A meaningful portion of demand is driven by hyperscalers and large cloud/enterprise customers with cyclical and quota-driven capex.

**Why it matters**  
- If hyperscaler capex slows (macro slowdown, AI prioritization changes, budget reallocations), NVIDIA’s Data Center revenue growth can decelerate quickly.
- AI infrastructure demand can be lumpy: customers may accelerate purchases in a given cycle and then pause/slow during capacity digestion.

**Key risk manifestations**
- Revenue growth moderation or negative growth during capex pauses.
- Order visibility declines; cancellations/rescheduling risk.
- Mix shift from “new cluster build” to “incremental refresh,” potentiall